In [2]:
from dotenv import load_dotenv
load_dotenv("/home/radha/Desktop/scfp125033_mca_lab/myenv/.env")

True

In [3]:
import os

In [4]:
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0
)

In [6]:
print("Groq LLM configured successfully!")

Groq LLM configured successfully!


In [7]:
transcript_text = """Interviewer: Tell me about a project where you used Python.
Candidate: I developed a student attendance application using Python.
I used functions to organize the code and handled invalid inputs using try-except blocks.

Interviewer: How do you work with databases?
Candidate: I have used SQL to retrieve and update records. I understand primary keys
and foreign keys, and I have practiced writing joins.

Interviewer: How do you debug a difficult problem?
Candidate: I reproduce the error, read the error message, and isolate the code causing
the issue. I then test a fix and check that the change does not break other functionality."""

In [8]:
skills_to_assess = [
    "Python programming",
    "Database and SQL knowledge",
    "Problem-solving and debugging",
    "Communication skills"
]

In [9]:
print("Interview transcript and assessment skills prepared!")

Interview transcript and assessment skills prepared!


In [10]:
class InterviewAnswer(BaseModel):
    question: str = Field(description="Interview question")
    answer: str = Field(description="Candidate's answer")

In [11]:
class ExtractedAnswers(BaseModel):
    answers: list[InterviewAnswer] = Field(
        description="List of interview questions and candidate answers"
    )

In [12]:
class SkillEvaluation(BaseModel):
    skill: str = Field(description="Skill being evaluated")
    demonstrated: bool = Field(
        description="True only when the transcript provides clear evidence of the skill"
    )
    evidence_or_gap: str = Field(
        description="Evidence from the candidate's answers or explanation of missing evidence"
    )

In [13]:
class SkillEvaluations(BaseModel):
    evaluations: list[SkillEvaluation] = Field(
        description="Evaluation of every requested skill"
    )

In [14]:
class ScorecardRow(BaseModel):
    skill: str = Field(description="Skill assessed")
    demonstrated: bool = Field(description="Whether the skill was clearly demonstrated")
    evidence_or_gap: str = Field(description="Evidence or missing evidence")
    rating: str = Field(description="Rating such as Demonstrated or Not Demonstrated")

In [15]:
class CandidateScorecard(BaseModel):
    scorecard_rows: list[ScorecardRow] = Field(
        description="Structured hiring scorecard containing one row per assessed skill"
    )
    overall_recommendation: str = Field(
        description="Overall recommendation, with a brief evidence-based explanation"
    )

In [16]:
print("All Pydantic output models created successfully!")

All Pydantic output models created successfully!


## Task1

In [17]:
answer_parser = PydanticOutputParser(
    pydantic_object=ExtractedAnswers
)

answer_prompt = PromptTemplate(
    template="""Extract every interview question and the candidate's corresponding answer
from the transcript.

Preserve the meaning of the candidate's answers.
Do not invent questions or answers.
Ignore unrelated interviewer comments.

Transcript:{transcript_text}

{format_instructions}""",
    input_variables=["transcript_text"],
    partial_variables={
        "format_instructions": answer_parser.get_format_instructions()
    }
)

In [18]:
answer_chain = answer_prompt | llm | answer_parser

In [19]:
extracted_answers = answer_chain.invoke({
    "transcript_text": transcript_text
})

In [20]:
print("STEP 1: EXTRACTED INTERVIEW ANSWERS")
print(extracted_answers.model_dump_json(indent=2))

STEP 1: EXTRACTED INTERVIEW ANSWERS
{
  "answers": [
    {
      "question": "Tell me about a project where you used Python.",
      "answer": "I developed a student attendance application using Python. I used functions to organize the code and handled invalid inputs using try-except blocks."
    },
    {
      "question": "How do you work with databases?",
      "answer": "I have used SQL to retrieve and update records. I understand primary keys and foreign keys, and I have practiced writing joins."
    },
    {
      "question": "How do you debug a difficult problem?",
      "answer": "I reproduce the error, read the error message, and isolate the code causing the issue. I then test a fix and check that the change does not break other functionality."
    }
  ]
}


## Task 2

In [26]:
skill_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a technical interviewer evaluating skill evidence.

Evaluate every requested skill using the extracted interview answers.

Rules:
- Evaluate every requested skill exactly once.
- Use only evidence from the extracted answers.
- Set demonstrated=True only when clear evidence exists.
- Set demonstrated=False when evidence is missing or insufficient.
- Explain the evidence or gap briefly.
- Do not invent experience or abilities.
- Missing evidence does not prove lack of ability.

{format_instructions}"""
    ),
    (
        "human",
        """Skills to assess:
{skills_to_assess}

Extracted interview answers:
{extracted_answers}"""
    )
]).partial(
    format_instructions=skill_parser.get_format_instructions()
)

print("Skill evaluation prompt created successfully!")

Skill evaluation prompt created successfully!


In [28]:
skill_chain = skill_prompt | llm | skill_parser

In [29]:
skill_evaluations = skill_chain.invoke({
    "skills_to_assess": ", ".join(skills_to_assess),
    "extracted_answers": extracted_answers.model_dump_json()
})

In [30]:
print("STEP 2: SKILL EVALUATION")
print(skill_evaluations.model_dump_json(indent=2))

STEP 2: SKILL EVALUATION
{
  "evaluations": [
    {
      "skill": "Python programming",
      "demonstrated": true,
      "evidence_or_gap": "Candidate described developing a student attendance application in Python, using functions to organize code and try-except blocks for input validation."
    },
    {
      "skill": "Database and SQL knowledge",
      "demonstrated": true,
      "evidence_or_gap": "Candidate stated using SQL to retrieve and update records, understanding primary and foreign keys, and practicing writing joins."
    },
    {
      "skill": "Problem-solving and debugging",
      "demonstrated": true,
      "evidence_or_gap": "Candidate explained a debugging process: reproducing the error, reading the error message, isolating problematic code, testing a fix, and verifying no regression."
    },
    {
      "skill": "Communication skills",
      "demonstrated": false,
      "evidence_or_gap": "The extracted answers do not provide evidence of communication skills such a

## Task 3

In [34]:
scorecard_prompt = ChatPromptTemplate.from_messages([
    ("system","""You are a hiring panel coordinator preparing a structured candidate scorecard.
Use only the supplied skill evaluations.
Include one row for every evaluated skill.
Preserve the demonstrated status and evidence or gap.
Use 'Demonstrated' when demonstrated=True and 'Not Demonstrated' otherwise.
Do not invent qualifications or evidence.
Explain the overall recommendation using the available evidence.
Missing evidence does not prove lack of ability.

{format_instructions}"""
    ),
    ("human","""Skill evaluations:
{skill_evaluations}"""
    )
]).partial(
    format_instructions=scorecard_parser.get_format_instructions()
)

print("Scorecard prompt created successfully!")

Scorecard prompt created successfully!


In [36]:
scorecard_chain = scorecard_prompt | llm | scorecard_parser

In [37]:
final_scorecard = scorecard_chain.invoke({
    "skill_evaluations": skill_evaluations.model_dump_json()
})

In [38]:
print("STEP 3: FINAL CANDIDATE SCORECARD")
print(final_scorecard.model_dump_json(indent=2))

STEP 3: FINAL CANDIDATE SCORECARD
{
  "scorecard_rows": [
    {
      "skill": "Python programming",
      "demonstrated": true,
      "evidence_or_gap": "Candidate described developing a student attendance application in Python, using functions to organize code and try-except blocks for input validation.",
      "rating": "Demonstrated"
    },
    {
      "skill": "Database and SQL knowledge",
      "demonstrated": true,
      "evidence_or_gap": "Candidate stated using SQL to retrieve and update records, understanding primary and foreign keys, and practicing writing joins.",
      "rating": "Demonstrated"
    },
    {
      "skill": "Problem-solving and debugging",
      "demonstrated": true,
      "evidence_or_gap": "Candidate explained a debugging process: reproducing the error, reading the error message, isolating problematic code, testing a fix, and verifying no regression.",
      "rating": "Demonstrated"
    },
    {
      "skill": "Communication skills",
      "demonstrated": f

In [41]:
print("=" * 50)
print("STEP 1: EXTRACTED ANSWERS")
print("=" * 50)
print(extracted_answers.model_dump_json(indent=2))

print("\n" + "=" * 50)
print("STEP 2: SKILL EVALUATION")
print("=" * 50)
print(skill_evaluations.model_dump_json(indent=2))

print("\n" + "=" * 50)
print("STEP 3: FINAL SCORECARD")
print("=" * 50)
print(final_scorecard.model_dump_json(indent=2))

STEP 1: EXTRACTED ANSWERS
{
  "answers": [
    {
      "question": "Tell me about a project where you used Python.",
      "answer": "I developed a student attendance application using Python. I used functions to organize the code and handled invalid inputs using try-except blocks."
    },
    {
      "question": "How do you work with databases?",
      "answer": "I have used SQL to retrieve and update records. I understand primary keys and foreign keys, and I have practiced writing joins."
    },
    {
      "question": "How do you debug a difficult problem?",
      "answer": "I reproduce the error, read the error message, and isolate the code causing the issue. I then test a fix and check that the change does not break other functionality."
    }
  ]
}

STEP 2: SKILL EVALUATION
{
  "evaluations": [
    {
      "skill": "Python programming",
      "demonstrated": true,
      "evidence_or_gap": "Candidate described developing a student attendance application in Python, using functions 